## **Setup**

In [1]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import gdown
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
import lightgbm as lgbm

import tensorflow as tf
import tensorflow_recommenders as tfrs
from tensorflow.keras.layers import StringLookup, TextVectorization, Embedding, GRU, Dense
from tensorflow.keras import layers

2025-11-18 19:12:16.768918: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-18 19:12:16.813391: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-18 19:12:18.169423: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## **Data preparation**

In [2]:
# Download data
file_id = "1GffOYmcAMP17oi2BwC7Dp4l5F7rEjHRr" 
url = f"https://drive.google.com/uc?id={file_id}"
output = "mind_large.zip"
is_downloaded = False

for file in os.listdir("."):
    if file.startswith("mind_large"):
        print("mind_large is already downloaded.")
        is_downloaded = True
        break
        
if is_downloaded == False:
    print("Downloading mind_large.zip ...")
    gdown.download(url, output, quiet=False)
    print("Download complete!")

    # Extract compressed file
    with zipfile.ZipFile("mind_large.zip", "r") as z:
        z.extractall(".")

mind_large is already downloaded.


In [3]:
# Sets
sets = ["train", "dev", "test"]

# News
news_header = ["id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
news = {}
for _set in sets:
    news[_set] = pd.read_csv(f"mind_large/news_{_set}.tsv", names=news_header, sep="\t")
all_news_df = pd.concat([news["train"], news["dev"], news["test"]], ignore_index=True)    

# Impressions
behaviors_header = ["impression_id", "user_id", "time", "history", "impressions"]
behaviors = {}
for _set in sets:
    behaviors[_set] = pd.read_csv(f"mind_large/behaviors_{_set}.tsv", names=behaviors_header, sep="\t")
all_behaviors_df = pd.concat([behaviors["train"], behaviors["dev"], behaviors["test"]], ignore_index=True)    

In [4]:
all_news_df = all_news_df.drop(columns=["url", "title_entities", "abstract_entities"])
all_news_df.drop_duplicates(inplace=True)
all_news_df["abstract"] = all_news_df["abstract"].fillna(all_news_df["title"])

In [5]:
all_behaviors_df["history"] = all_behaviors_df["history"].fillna("")

## **Filtering data (to keep relevant rows)**

In [6]:
len(all_behaviors_df)

4979946

In [7]:
behaviors_df = all_behaviors_df[
    (all_behaviors_df["impressions"].str.contains("-1")) & # Keep only impressions with at least one click
    (all_behaviors_df["history"].str.len() > 0) & # Keep users with AT LEAST ONE item in their history
    (all_behaviors_df["impressions"].str.len() >= 10) # Keep only impressions with at least 2 items shown
]

len(behaviors_df)

2551884

In [8]:
# Sampling (my computer is not that performant :))
behaviors_df = behaviors_df.sample(n=20_000)

## **Building the user-news interaction dataset**

In [9]:
# Function to split the impressions and clicks into two separate lists
def process_impression(impression_list):
    clicked, non_clicked = [], []
    if impression_list != "":
        list_of_strings = impression_list.split()
        clicked = [x.split("-")[0] for x in list_of_strings if x.split("-")[1] == "1"]
        non_clicked = [x.split("-")[0] for x in list_of_strings if x.split("-")[1] == "0"]
    return clicked, non_clicked

In [10]:
# Separate views from clicks
behaviors_df[["clicked", "non_clicked"]] = behaviors_df["impressions"].apply(
    lambda x: pd.Series(process_impression(x))
)
# Split history
behaviors_df["history"] = behaviors_df["history"].apply(lambda x: x.split())

behaviors_df.head()

,impression_id,user_id,time,history,impressions,clicked,non_clicked
2098437,2098438,U133297,11/11/2019 1:12:14 PM,"[N127259, N81899, N129530, N10737, N48598, N89...",N85452-0 N54015-1 N25967-0 N46420-0 N83702-0 N...,[N54015],"[N85452, N25967, N46420, N83702, N28009, N1230..."
2382354,149607,U503808,11/15/2019 12:11:17 PM,"[N114369, N15819, N116115, N99426, N70777, N10...",N58465-0 N18468-0 N55210-0 N36134-0 N116564-0 ...,[N102499],"[N58465, N18468, N55210, N36134, N116564, N394..."
2109815,2109816,U634415,11/12/2019 1:34:32 PM,"[N91231, N108143, N67790, N3176, N71728, N1013...",N105071-1 N7937-0 N33078-0 N110303-0,[N105071],"[N7937, N33078, N110303]"
2526386,293639,U680136,11/15/2019 8:41:03 AM,"[N251, N129790, N104200, N82060, N128965, N100...",N72977-0 N12974-0 N18190-0 N54368-0 N122944-0 ...,[N35304],"[N72977, N12974, N18190, N54368, N122944, N584..."
2473370,240623,U744102,11/15/2019 5:27:27 AM,"[N42703, N41143]",N44453-0 N23748-0 N16989-0 N68048-0 N59052-0 N...,[N35083],"[N44453, N23748, N16989, N68048, N59052, N3098..."


In [11]:
%%time

click_data = []
for _, row in behaviors_df.iterrows():
    history = row["history"]
    clicked_news, non_clicked_news = row["clicked"], row["non_clicked"]
    
    for news_id in clicked_news:
        click_data.append({
            "history": history,
            "candidate_news_id": news_id,
            "label": 1
        })
        
    # We can also sample non-clicked news for harder negatives
    for news_id in non_clicked_news:
         click_data.append({
            "history": history,
            "candidate_news_id": news_id,
            "label": 0
        })

# Create a DataFrame from the exploded data
training_df = pd.DataFrame(click_data)
training_df.head()

CPU times: user 1.83 s, sys: 74 ms, total: 1.91 s
Wall time: 1.91 s


,history,candidate_news_id,label
0,"[N127259, N81899, N129530, N10737, N48598, N89...",N54015,1
1,"[N127259, N81899, N129530, N10737, N48598, N89...",N85452,0
2,"[N127259, N81899, N129530, N10737, N48598, N89...",N25967,0
3,"[N127259, N81899, N129530, N10737, N48598, N89...",N46420,0
4,"[N127259, N81899, N129530, N10737, N48598, N89...",N83702,0


In [12]:
# For Retrieval (Two-Tower): We only need positive interactions (label=1)
retrieval_df = training_df[training_df["label"] == 1].copy()

# This will be used to join features to the candidate_news_id
news_features_df = all_news_df[["id", "category", "title"]].copy()
news_features_df = news_features_df.rename(columns={"id": "candidate_news_id"})

# Merge retrieval_df with all_news_df
retrieval_df_merged = retrieval_df.merge(
    news_features_df,
    on="candidate_news_id",
    how="left"
)

# We now use the merged DataFrame which contains all the required features.
# We also use the keys that compute_loss expects ("news_id", "category", "title")
retrieval_ds = tf.data.Dataset.from_tensor_slices({
    "history": tf.ragged.constant(retrieval_df_merged["history"].values),
    "news_id": tf.constant(retrieval_df_merged["candidate_news_id"].values),
    "category": tf.constant(retrieval_df_merged["category"].values),
    "title": tf.constant(retrieval_df_merged["title"].values)
})

print(f"Created {len(retrieval_df_merged)} positive pairs for retrieval training.")

Created 30283 positive pairs for retrieval training.


E0000 00:00:1763489585.305026  141633 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1763489585.315500  141633 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2025-11-18 19:13:05.317331: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 29906280 exceeds 10% of free system memory.


## **Stage 1: Retrieval (Two-Tower Model)**

**Defining the hyperparameters for the training**

In [13]:
EMBEDDING_DIM = 64 # Output embeddings dimension for both User and News tower
MAX_HISTORY_LENGTH = 30 # Max number of articles to look at in user history
MAX_TOKENS = 20000 # Max vocab size for titles
TITLE_VECTORIZATION_DIM = 100

**Building the News Tower**

This model turns a NewsID into an embedding

In [14]:
class NewsModel(tf.keras.Model):
    
    
    def __init__(self, all_news_ids, all_categories, **kwargs):
        super().__init__(**kwargs)
        self.all_news_ids = all_news_ids
        self.all_categories = all_categories
        # News ID embedding model
        self.news_id_lookup = StringLookup(vocabulary=self.all_news_ids, mask_token=None)
        self.news_id_embedding_model = Embedding(input_dim=len(self.all_news_ids) + 1, output_dim=EMBEDDING_DIM)
        # Category embedding model
        self.category_lookup = StringLookup(vocabulary=self.all_categories, mask_token=None)
        self.category_embedding_model = Embedding(input_dim=len(self.all_categories) + 1, output_dim=EMBEDDING_DIM)
        # Title vectorizer
        self.title_vectorizer = TextVectorization(
            max_tokens=MAX_TOKENS,
            output_mode="int",
            output_sequence_length=TITLE_VECTORIZATION_DIM
        )
        # Title embedding model
        self.title_embedding_model = tf.keras.Sequential([
            Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM),
            tf.keras.layers.GlobalAveragePooling1D() # Average word embeddings
        ])
        # Final Dense Layer
        self.dense = Dense(EMBEDDING_DIM)
                    
    
    def call(self, inputs):
        # Get embedding for each feature
        news_id_embedding = self.news_id_embedding_model(self.news_id_lookup(inputs["news_id"]))
        category_embedding = self.category_embedding_model(self.category_lookup(inputs["category"]))
        title_embedding = self.title_embedding_model(self.title_vectorizer(inputs["title"]))
        # Combine them
        combined_embeddings = tf.concat([news_id_embedding, category_embedding, title_embedding], axis=1)
        # Pass through the final dense layer to get a single 64-dim vector
        return self.dense(combined_embeddings)
    
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "all_news_ids": self.all_news_ids,
            "all_categories": self.all_categories,
        })
        return config
    
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)

**Build the User (Session) Tower**

This model turns a user's click history into an embedding

In [15]:
class UserModel(tf.keras.Model):
    
    
    def __init__(self, news_id_embedding_model, all_news_ids, **kwargs):
        super().__init__(**kwargs)
        # Use the same embedding layer as the news model        
        self.news_id_embedding_model = news_id_embedding_model
        self.all_news_ids = all_news_ids
        self.news_id_lookup = StringLookup(vocabulary=self.all_news_ids, mask_token=None)
        # We use a GRU to process the sequence of clicked news
        self.gru = GRU(EMBEDDING_DIM)
        

    def call(self, history):
        history = history[:, -MAX_HISTORY_LENGTH:]
        history_int = self.news_id_lookup(history)
        # Get embeddings for each news ID in the history
        history_embeddings = self.news_id_embedding_model(history_int)
        # Output
        return self.gru(history_embeddings)
    
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "all_news_ids": self.all_news_ids,
            "news_id_embedding_model": tf.keras.utils.serialize_keras_object(self.news_id_embedding_model),
        })
        return config
    
    
    @classmethod
    def from_config(cls, config):
        news_id_embedding_model_config = config.pop("news_id_embedding_model")
        news_id_embedding_model = tf.keras.utils.deserialize_keras_object(news_id_embedding_model_config)        
        return cls(news_id_embedding_model=news_id_embedding_model, **config)

**Combining the two towers**

In [16]:
class MINDRetrievalModel(tfrs.Model):
    
    
    def __init__(self, user_model, news_model, candidate_dataset):
        super().__init__()
        self.user_model = user_model
        self.news_model = news_model
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=candidate_dataset.map(self.news_model),
                    ks=[20]
            )
        )    
    
    
    def compute_loss(self, features, training=False):
        user_embeddings = self.user_model(features["history"])
        # Create the dictionary for the news model
        news_features = {
            "news_id": features["news_id"],
            "category": features["category"],
            "title": features["title"]
        }
        positive_news_embeddings = self.news_model(news_features)
        return self.task(user_embeddings, positive_news_embeddings)

**Training the model**

In [17]:
# Define datasets
train_ds = retrieval_ds.take(18_000)
val_ds = retrieval_ds.skip(18_000).take(1_000)
test_ds = retrieval_ds.skip(19_000)

# Batch data
train_ds_batched = train_ds.shuffle(18_000).batch(128).cache()
val_ds_batched = val_ds.batch(128).cache()
test_ds_batched = test_ds.batch(128).cache()

# Use for computing metrics in the Two-Tower
news_ds = tf.data.Dataset.from_tensor_slices({
    "news_id": all_news_df["id"].values,
    "category": all_news_df["category"].values,
    "title": all_news_df["title"].values
})

In [18]:
# Vocabulary for categorical features (id + category)
all_news_ids = all_news_df["id"].unique()
all_categories = all_news_df["category"].unique()
news_ids_list = all_news_ids.tolist() if hasattr(all_news_ids, "tolist") else all_news_ids
categories_list = all_categories.tolist() if hasattr(all_categories, "tolist") else all_categories

# News model
news_model = NewsModel(all_news_ids=news_ids_list, all_categories=categories_list)
news_model.title_vectorizer.adapt(all_news_df["title"])

# User model
user_model = UserModel(news_id_embedding_model=news_model.news_id_embedding_model,
                       all_news_ids=news_ids_list)

# Retrieval task
sampled_candidate_dataset = news_ds.batch(128).take(400) # We just take a little part to avoid computation issues
retrieval_model = MINDRetrievalModel(user_model=user_model, news_model=news_model, candidate_dataset=sampled_candidate_dataset)
retrieval_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

2025-11-18 19:13:09.819786: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.
2025-11-18 19:13:09.826435: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.
2025-11-18 19:13:09.833478: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.


In [19]:
# Train for a few epochs
history = retrieval_model.fit(train_ds_batched,
                              validation_data=val_ds_batched,
                              epochs=10)

Epoch 1/10


2025-11-18 19:13:12.514137: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.


141/141 [==============================] - 90s 610ms/step - factorized_top_k/top_20_categorical_accuracy: 0.0097 - loss: 615.0031 - regularization_loss: 0.0000e+00 - total_loss: 615.0031 - val_factorized_top_k/top_20_categorical_accuracy: 0.0480 - val_loss: 478.8054 - val_regularization_loss: 0.0000e+00 - val_total_loss: 478.8054
Epoch 2/10
141/141 [==============================] - 87s 619ms/step - factorized_top_k/top_20_categorical_accuracy: 0.0772 - loss: 591.7657 - regularization_loss: 0.0000e+00 - total_loss: 591.7657 - val_factorized_top_k/top_20_categorical_accuracy: 0.0650 - val_loss: 478.6945 - val_regularization_loss: 0.0000e+00 - val_total_loss: 478.6945
Epoch 3/10
141/141 [==============================] - 86s 610ms/step - factorized_top_k/top_20_categorical_accuracy: 0.1266 - loss: 564.0391 - regularization_loss: 0.0000e+00 - total_loss: 564.0391 - val_factorized_top_k/top_20_categorical_accuracy: 0.0580 - val_loss: 493.8874 - val_regularization_loss: 0.0000e+00 - val_tot

In [20]:
# Evaluate the model
metrics = retrieval_model.evaluate(test_ds_batched, return_dict=True)
metrics

89/89 [==============================] - 36s 409ms/step - factorized_top_k/top_20_categorical_accuracy: 0.0946 - loss: 975.2062 - regularization_loss: 0.0000e+00 - total_loss: 975.2062


{'factorized_top_k/top_20_categorical_accuracy': 0.09456704556941986,
 'loss': 106.7613754272461,
 'regularization_loss': 0,
 'total_loss': 106.7613754272461}

In [21]:
# Save the models for inference 

# We need the user tower to get user embeddings and )
user_model.save("models/user_retrieval_model", save_format="tf")

# We need the news tower to build the candidate index
news_model.save("models/news_retrieval_model", save_format="tf")

INFO:tensorflow:Assets written to: models/user_retrieval_model/assets


INFO:tensorflow:Assets written to: models/user_retrieval_model/assets


INFO:tensorflow:Assets written to: models/news_retrieval_model/assets


INFO:tensorflow:Assets written to: models/news_retrieval_model/assets


In [36]:
# Create and save scann index for fast retrieval

scann_index = tfrs.layers.factorized_top_k.ScaNN(k=200)
scann_index.index_from_dataset(
    tf.data.Dataset.zip((news_ds.batch(128).map(lambda x: x["news_id"]), news_ds.batch(128).map(news_model)))
)
dummy_embedding = tf.random.uniform(shape=[1, EMBEDDING_DIM], dtype=tf.float32)

_ = scann_index(dummy_embedding)
tf.saved_model.save(scann_index, "models/scann_index", options=tf.saved_model.SaveOptions(namespace_whitelist=["Scann"]))

I0000 00:00:1763491141.053246  141633 partitioner_factory_base.cc:58] Size of sampled dataset for training partition: 100000
I0000 00:00:1763491141.128007  141633 kmeans_tree_partitioner_utils.h:90] PartitionerFactory ran in 74.731816ms.


INFO:tensorflow:Assets written to: models/scann_index/assets


INFO:tensorflow:Assets written to: models/scann_index/assets


## **Stage 2: Ranking (GDBT Model)**

In [23]:
# Load the trained embedding models

user_model = tf.keras.models.load_model("models/user_retrieval_model")
news_model = tf.keras.models.load_model("models/news_retrieval_model")

In [24]:
# Create a full news embedding lookup (dictionary)

all_news_embeddings = {}
for news_id_batch in news_ds.batch(512):
    embeddings_batch = news_model(news_id_batch)
    news_ids = news_id_batch["news_id"].numpy()
    for news_id, embedding in zip(news_ids, embeddings_batch.numpy()):
        all_news_embeddings[news_id.decode("utf-8")] = embedding

2025-11-18 19:28:03.369549: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [25]:
# Create user embedding cache
user_embeddings_cache = {}

# Helper function to get features
def get_features(row):
    features = {}
    
    # Get user history
    user_history = row["history"]
    history_key = tuple(user_history) # Use tuple as dict key
    
    # Get user embedding
    if history_key in user_embeddings_cache:
        user_emb = user_embeddings_cache[history_key]
    else:
        user_emb = user_model(tf.ragged.constant([user_history])).numpy()[0]
        user_embeddings_cache[history_key] = user_emb
        
    # Get candidate news embedding
    candidate_id = row["candidate_news_id"]
    candidate_emb = all_news_embeddings.get(candidate_id, np.zeros(EMBEDDING_DIM))
    
    # Get some features for the model
    
    # 1. Retrieval Score (dot product)
    features["retrieval_dot_product"] = np.dot(user_emb, candidate_emb)
    
    # 2. User Features
    features["history_length"] = len(user_history)
    
    # 3. Candidate News Features (lookup from news_df)
    news_info = all_news_df.loc[all_news_df["id"] == candidate_id].iloc[0]
    features["category"] = news_model.category_lookup(news_info["category"]).numpy()
    
    # 4. Cross Features
    # How many times this category appeared in history ?
    history_categories = all_news_df[all_news_df["id"].isin(user_history)]["category"].values
    features["category_in_history_count"] = np.sum(history_categories == news_info["category"])
    
    return features

In [26]:
# Training data

ranking_sample_df = training_df.sample(n=2_000, random_state=42)

features_list = ranking_sample_df.apply(get_features, axis=1)
X = pd.DataFrame.from_records(features_list.tolist())
y = ranking_sample_df["label"]

# Convert categorical features for LightGBM
X["category"] = X["category"].astype("category")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [27]:
X_train.head()

,retrieval_dot_product,history_length,category,category_in_history_count
968,-2.712502,25,1,7
240,4.023776,9,9,0
819,-6.635718,32,2,11
692,-4.864827,7,4,0
420,-6.964602,31,7,4


In [28]:
# Train the model

lgbm_train = lgbm.Dataset(X_train, y_train, categorical_feature=["category"])
lgbm_eval = lgbm.Dataset(X_test, y_test, reference=lgbm_train, categorical_feature=["category"])

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "verbose": -1
}

lgbm_model = lgbm.train(
    params,
    lgbm_train,
    num_boost_round=500,
    callbacks=[lgbm.early_stopping(10, verbose=False)],
    valid_sets=[lgbm_eval]
)

In [29]:
# Save the model

lgbm_model.save_model("models/ranking_model.txt")

## **Stage 3: Re-ranking (Maximal Marginal Relevance - MMR)**

In [30]:
def mmr_rerank(candidate_ids, gbdt_scores, item_embeddings_dict, lambda_val=0.5, k=10):
    """
    Performs MMR re-ranking.
    
    Args:
        candidate_ids (list): List of news IDs, sorted by GBDT score (highest first).
        gbdt_scores (list): List of GBDT scores corresponding to candidate_ids.
        item_embeddings_dict (dict): {news_id: embedding} lookup.
        lambda_val (float): 0.5 = balanced relevance/diversity. 1.0 = GBDT rank. 0.0 = diversity only.
        k (int): Number of items to return.
    
    Returns:
        list: The re-ranked list of k news IDs.
    """
    
    # Zip candidates and scores, keep sorted by score
    candidates = sorted(zip(candidate_ids, gbdt_scores), key=lambda x: x[1], reverse=True)
    
    # Store embeddings for quick lookup
    candidate_embeddings = {cid: item_embeddings_dict.get(cid) for cid, _ in candidates}
    
    # Filter out any candidates we don't have embeddings for
    candidates = [(cid, score) for cid, score in candidates if candidate_embeddings.get(cid) is not None]

    if not candidates:
        return []

    selected = []
    selected_embeddings = []
    
    # Add the highest-scoring item first
    top_candidate_id, top_score = candidates.pop(0)
    selected.append(top_candidate_id)
    selected_embeddings.append(candidate_embeddings[top_candidate_id])
    
    while len(selected) < k and candidates:
        best_item_id = None
        best_mmr_score = -np.inf
        
        items_to_remove_idx = -1
        
        # Iterate over remaining candidates
        for i, (candidate_id, score) in enumerate(candidates):
            candidate_emb = candidate_embeddings[candidate_id]
            
            # Calculate similarity to already selected items
            # Shape: (1, embed_dim) vs (len(selected), embed_dim)
            sim_to_selected = cosine_similarity([candidate_emb], selected_embeddings)
            max_sim = np.max(sim_to_selected) # The "marginal" part
            
            # MMR Score = lambda * (Relevance) - (1 - lambda) * (Max Similarity)
            mmr_score = lambda_val * score - (1 - lambda_val) * max_sim
            
            if mmr_score > best_mmr_score:
                best_mmr_score = mmr_score
                best_item_id = candidate_id
                items_to_remove_idx = i
        
        if best_item_id:
            selected.append(best_item_id)
            selected_embeddings.append(candidate_embeddings[best_item_id])
            candidates.pop(items_to_remove_idx)
        else:
            # Should not happen if candidates list is not empty
            break
            
    return selected

## **Stage 4: The Inference Pipeline**

In [31]:
# Load all models (towers, indexes, ranking)

user_retrieval_model = tf.keras.models.load_model("models/user_retrieval_model") # Retrieval
scann_index = tf.saved_model.load("models/scann_index") # Retrieval
ranking_model = lgbm.Booster(model_file="models/ranking_model.txt") # Ranking

In [37]:
# Getting recommendations for a specific user

def get_session_recommendations(user_id, history_news_ids, k=10):
    
    print(f"\nGetting recommendations for User {user_id} ...")
    
    # Stage 1: Retrieval (Get top 200 candidates)
    # User embedding
    user_history_tensor = tf.ragged.constant([history_news_ids])
    
    user_embedding = user_retrieval_model(user_history_tensor)
    # Use the ScaNN index to get top 200 candidates
    scores, retrieved_ids = scann_index(user_embedding)
    retrieved_ids = retrieved_ids.numpy()[0].astype(str)
    print(f"Stage 1 (Retrieval): Found {len(retrieved_ids)} candidates.")


    # Stage 2: Ranking
    # Create feature vectors for the 200 candidates
    ranking_features = []
    valid_candidate_ids = []
    
    # Get user embedding once (as numpy)
    user_emb_np = user_embedding.numpy()[0]
    
    for news_id in retrieved_ids:
        # Get candidate embedding
        candidate_emb_np = all_news_embeddings.get(news_id)
        if candidate_emb_np is None:
            continue # Skip if we have no embedding
            
        # Feature engineering (must match training!)
        features = {}
        features["retrieval_dot_product"] = np.dot(user_emb_np, candidate_emb_np)
        features["history_length"] = len(history_news_ids)
        
        news_info = all_news_df.loc[all_news_df["id"] == news_id]
        if news_info.empty:
            continue
        news_info = news_info.iloc[0]
            
        features["category"] = news_model.category_lookup(news_info["category"]).numpy()
        history_categories = all_news_df[all_news_df["id"].isin(history_news_ids)]["category"].values
        features["category_in_history_count"] = np.sum(history_categories == news_info["category"])
        
        ranking_features.append(features)
        valid_candidate_ids.append(news_id)

    if not ranking_features:
        print("No valid candidates after feature engineering.")
        return []

    # Create DataFrame for GBDT
    X_rank = pd.DataFrame.from_records(ranking_features)
    X_rank["category"] = X_rank["category"].astype("category")
    
    # Get scores from GBDT
    gbdt_scores = ranking_model.predict(X_rank, num_iteration=ranking_model.best_iteration)
    print(f"Stage 2 (Ranking): Scored {len(gbdt_scores)} candidates.")


    # Stage 3: Re-ranking
    # Take top 30 from GBDT to re-rank for diversity
    top_candidates_df = pd.DataFrame({
        "news_id": valid_candidate_ids,
        "score": gbdt_scores
    }).nlargest(30, "score")
    
    final_recommendations = mmr_rerank(
        candidate_ids=top_candidates_df["news_id"].tolist(),
        gbdt_scores=top_candidates_df["score"].tolist(),
        item_embeddings_dict=all_news_embeddings,
        lambda_val=0.7, # Favor relevance a bit more
        k=k
    )
    print(f"Stage 3 (Re-ranking): Produced final {len(final_recommendations)} recommendations.")
    return final_recommendations

In [38]:
# Example
sample_user = behaviors_df.iloc[100]
user_history = sample_user["history"]

recs = get_session_recommendations(sample_user["user_id"], user_history)

print("Final recommendations")
for i, news_id in enumerate(recs):
    news_title = all_news_df[all_news_df["id"] == news_id].iloc[0]["title"]
    print(f"{i+1}. {news_id} - {news_title}")


Getting recommendations for User U640747 ...
Stage 1 (Retrieval): Found 200 candidates.
Stage 2 (Ranking): Scored 200 candidates.
Stage 3 (Re-ranking): Produced final 10 recommendations.
Final recommendations
1. N75838 - Save the Planet (Except the Babies)
2. N44077 - Browns S Damarious Randall ejected for brutal helmet-to-helmet hit on Diontae Johnson
3. N42433 - Trump could slap steep tariffs on imported cars next week
4. N127679 - The environmental toll of cremating the dead
5. N31094 - Trump attacks ambassador on Twitter as she testifies that his words in Ukraine call made her feel threatened
6. N98052 - 2 suspects in custody after stolen vehicle crashed into North County home
7. N81448 - Search For Missing Florida Girl Finds Human Remains
8. N129602 - Cold blast to bring single-digit wind chills for Midwest, Northeast
9. N109881 - Map of hypertension hot spots in Minnesota reveals surprises
10. N110667 - Officials: At least 13 dead in Slovakia bus crash


## **Resources**

* https://www.tensorflow.org/recommenders/examples/basic_retrieval
* https://www.kaggle.com/code/jacobwelander/mind-recommender-from-scratch-2023
* https://www.kaggle.com/code/kanruwang/tensorflow-recommender-two-tower-multitask